In [1]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from langchain_core.runnables import RunnableLambda

# 1. Load environment variables first
load_dotenv()
parser = StrOutputParser()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
URL = os.getenv("URL")
API_KEY = os.getenv("APIKEY")


C:\Users\Personal\PycharmProjects\NLP-RAG-BY-HOPETOSKILL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

KeyboardInterrupt



In [83]:
from langchain_google_genai import ChatGoogleGenerativeAI


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=GROQ_API_KEY)


In [82]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2781.02it/s]


**To Delete The QDRANT Collection**

In [24]:
# from qdrant_client import QdrantClient
#
# client = QdrantClient(url=URL, api_key=API_KEY)
#
# # delete the collection
# client.delete_collection("vanilla_rag")
#
# print("Collection deleted")

Collection deleted


In [84]:
# 3. File Loading
txtfile_path = "yarvalley.txt"
loader = TextLoader(txtfile_path)
txtfile = loader.load()


In [85]:
# 4. Text Splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=150,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)
split_document = splitter.split_documents(txtfile)

In [87]:
client = QdrantClient(
    url=URL,
    api_key=API_KEY
)

collections = client.get_collections().collections

collection_exists = any(
    collection.name == "vanilla_rag"
    for collection in collections
)

# =========================
# 7. Create Collection Only If Not Exists
# =========================

if not collection_exists:

    vectorstore = QdrantVectorStore.from_documents(
        documents=split_document,
        embedding=embeddings,
        url=URL,
        api_key=API_KEY,
        collection_name="vanilla_rag",
    )

    print("Collection created and documents stored.")

else:

    vectorstore = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        url=URL,
        api_key=API_KEY,
        collection_name="vanilla_rag",
    )

    print("Collection already exists. Using existing collection.")

Collection already exists. Using existing collection.


In [88]:
retriever = vectorstore.as_retriever(search_kwargs={"k":7})

rag_prompt = ChatPromptTemplate.from_template(
    """
    Please answer the following questions,if it is in context otherwise answer "I don,t have enough information about this"
    Context:
    {context}

    Question:
    {question}


    """
)

In [89]:
#This return objects in a string format
def get_content(docs):
    return "\n\n".join( doc.page_content for doc in docs )

In [90]:
#context and question will fill this(rag_prompt)
#LCEL

rag_chain = (
    {
        "context": retriever | get_content,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser)

In [91]:
question = "Where is malam jabba in swat"
answer = rag_chain.invoke(question)

print(answer)

Malam Jabba is located about 44 km from Mingora in Swat.


**RAGAS(RAG Assesment)**

In [92]:
test_questions = [
    "Where is Swat?",
    "Where is Malam Jabba in Swat?",
    "What is Swat famous for?",
    "How many tourist spots are there in Swat?",
]

ground_truths = [
    "Swat Valley is located in the Malakand Division of Khyber Pakhtunkhwa province of Pakistan, situated north of Peshawar between 34°40' to 35°N latitude and 72° to 74°6'E longitude.",

    "Malam Jabba is located about 44 km from Mingora in Swat Valley. It is a modern hill resort featuring snowy mountain peaks, green valleys, forests, a chairlift, and a ski resort restored by TCKP in 2015.",

    "Swat is famous for its scenic natural beauty earning it the title Switzerland of the East, Buddhist civilization remnants and Gandhara art, emerald mines near Mingora, Malam Jabba ski resort, and tourist spots like Kalam, Bahrain, Madyan, Marghuzar, and Miandam.",

    "The document does not provide a specific numerical count of tourist spots, but it explicitly names and describes several key destinations and valleys in Swat, including: Mingora, Saidu Sharif(including the Swat Museum, the Tomb of the Akhund of Swat, and the Butkara archeological sites), Malam Jabba Ski Resort, Marghuzar, Miandam, Madyan, Bishigram Valley, Bahrain, Kalam (including Ushu, Utrot, Gabral, and Mahodand Lake), and Mankial Valley.",
]

In [93]:
answers = []
contexts = []

for question in test_questions:
    # get answer from LLM
    answer = rag_chain.invoke(question)
    answers.append(answer)

    # get contexts from Qdrant
    retrieved_docs = retriever.invoke(question) #Dear retriever goto qdrant store and search against this question and give me content for this question
    retrieved_docs_content = [doc.page_content for doc in retrieved_docs]
    contexts.append(retrieved_docs_content)

In [94]:
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision, context_recall


In [95]:
data = [
    {
        "user_input"  : test_questions[0],
        "response"    : answers[0],
        "retrieved_contexts" : contexts[0],
        "reference"   : ground_truths[0],
    },
    {
        "user_input"  : test_questions[1],
        "response"    : answers[1],
        "retrieved_contexts" : contexts[1],
        "reference"   : ground_truths[1],
    },
    {
        "user_input"  : test_questions[2],
        "response"    : answers[2],
        "retrieved_contexts" : contexts[2],
        "reference"   : ground_truths[2],
    },
    {
        "user_input"  : test_questions[3],
        "response"    : answers[3],
        "retrieved_contexts" : contexts[3],
        "reference"   : ground_truths[3],
    },
]

dataset = EvaluationDataset.from_list(data)

In [96]:
judge_llm = LangchainLLMWrapper(llm)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\Lenovo yoga\AppData\Local\Temp\ipykernel_19264\987134942.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(llm)
C:\Users\Lenovo yoga\AppData\Local\Temp\ipykernel_19264\987134942.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [97]:
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

from ragas.run_config import RunConfig


# set llm on each metric manually

f = Faithfulness()
ar = AnswerRelevancy()
cp = ContextPrecision()
cr = ContextRecall()

f.llm = judge_llm
ar.llm = judge_llm
cp.llm = judge_llm
cr.llm = judge_llm

ar.embeddings = judge_embeddings  # answer relevancy also needs embeddings

result = evaluate(
    dataset=dataset,
    metrics=[f, ar, cp, cr],
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=RunConfig(max_workers=1, timeout=180)
)

print(result)

C:\Users\Lenovo yoga\AppData\Local\Temp\ipykernel_19264\4094472146.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Lenovo yoga\AppData\Local\Temp\ipykernel_19264\4094472146.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Lenovo yoga\AppData\Local\Temp\ipykernel_19264\4094472146.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exam

{'faithfulness': 0.5167, 'answer_relevancy': nan, 'context_precision': 0.2857, 'context_recall': 0.6750}
